# Suite CON — Predicates

A predicate is a rule over symbols, each bound to an object of a schema, and parameters, each a value; it applies to a
match, which binds every symbol. Predicates are terms of the predicate algebra, built fluently as schemas are, and are
data: applied by reference, written with their symbols' schema names, and read back through a store that resolves them.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Terms } from "@mbse/expressions/Framework";
import { Constraints as C, Predicates as P, register } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Proxies, Schemas as S, Validators as SV } from "@mbse/schemas/Framework";
import { Contact, Phone, assert, book, equal, raises, same } from "./support.js";

const the = E.variable("the");
const IsAnAdult = new P.OfPredicate.Builder()
  .name("IsAnAdult")
  .description("18 or older")
  .symbols({ the: Contact })
  .requires(the.age.ge(18n))
  .create();
const Numbered = new P.OfPredicate.Builder().name("Numbered").symbols({ the: Phone }).requires(the.has("number")).create();

## CON-01 · A predicate is built fluently: a name, a description, symbols, parameters and conditions

In [ ]:
{
  assert(IsAnAdult.name === "IsAnAdult" && IsAnAdult.description === "18 or older" && same(IsAnAdult.symbols.keys(), ["the"])
    && IsAnAdult.parameters.size === 0);
  assert(IsAnAdult.symbols.get("the") === Contact && IsAnAdult.rule instanceof E.OfOperation.Data); // a writer, resolved
  assert(Terms.same(IsAnAdult.rule, the.age.ge(18n).data));
  assert(Numbered.description === null && IsAnAdult.validate({ core: true }).length === 0 && same(IsAnAdult.binds(), ["the"]));
  const two = new P.OfPredicate.Builder().symbols({ a: Contact }).symbols(new Map([["b", Phone]])).parameters(
    (p) => p.name("n").of((t) => t.as_native(BigInt)), (p) => p.name("any")).create();
  assert(same(two.symbols.keys(), ["a", "b"]) && two.rule === null); // symbols add up, in order; nothing validates yet
  assert(same(two.parameters.keys(), ["n", "any"]) && two.parameters.get("any") === null && same(two.binds(), ["a", "b", "n", "any"]));
  const renamed = new P.OfPredicate.Builder(IsAnAdult).name("IsGrown").clone();
  assert(renamed.name === "IsGrown" && IsAnAdult.name === "IsAnAdult" && renamed.rule === IsAnAdult.rule);
  assert(equal([...renamed.symbols], [...IsAnAdult.symbols]) && renamed.symbols !== IsAnAdult.symbols);
  assert(new P.OfPredicate.Builder(renamed).description("grown").update() === renamed && renamed.description === "grown");
  const both = new P.OfPredicate.Builder(IsAnAdult).requires(the.has("name")).forbids(the.has("pager")).clone(); // conditions add up
  assert(Terms.same(both.rule, the.age.ge(18n).and_(the.has("name")).and_(the.has("pager").not_()).data));
  assert(IsAnAdult.rule !== both.rule);
  assert(Terms.same(IsAnAdult, new P.OfPredicate.Builder(IsAnAdult).clone())); // equal symbols compare equal
  assert(!Terms.same(IsAnAdult, new P.OfPredicate(IsAnAdult.name, IsAnAdult.description, IsAnAdult.rule, new P.Symbols([["the", Phone]]))));
  assert(!Terms.same(IsAnAdult, new P.OfPredicate(IsAnAdult.name, IsAnAdult.description, IsAnAdult.rule, new P.Symbols([["x", Contact]]))));
  assert(!Terms.same(IsAnAdult, new P.OfPredicate.Builder(IsAnAdult).symbols({ x: Phone }).clone())); // one more
  assert(!new P.Symbols().equals(new Map())); // TypeScript only: Symbols equal only Symbols
  const builder = new P.OfPredicate.Builder().symbols({ c: Contact }).parameters((p) => p.name("least"));
  assert(Terms.same(builder.c.data, E.variable("c").data) && Terms.same(builder.least.data, E.variable("least").data)); // declared
  assert(builder.nope === undefined); // TypeScript gives undefined where Python raises AttributeError
  assert(P.OfPredicate.resolve(IsAnAdult) === IsAnAdult);
  assert(P.OfPredicate.resolve((b: P.OfPredicate.Builder) => b.name("X").symbols({ x: Contact }).requires(true)).name === "X");
  raises(TypeError, () => P.OfPredicate.resolve("X"), "expected a predicate or a callable taking its builder, got 'X'");
  raises(Errors.ValueError, () => new P.OfPredicate.Builder(IsAnAdult).create(),
    "create() is only valid without a source instance; use clone() or update()");
  raises(Errors.ValueError, () => new P.OfPredicate.Builder().clone(), "clone() is only valid with a source instance");
  raises(Errors.ValueError, () => new P.OfPredicate.Builder().update(), "update() is only valid with a source instance");
  assert(IsAnAdult.schema_name() === "Patterns.Predicate" && IsAnAdult.owner() === null && IsAnAdult.identity() === IsAnAdult.identity());
}

## CON-02 · Predicates are checked when asked

In [ ]:
{
  const Value = new S.OfObject.Builder().name("Value").create();
  const bad = new P.OfPredicate.Builder().name("Bad").symbols({ a: Value, b: new S.OfObject.Builder().ref().create() }).requires(
    E.operation("pow", E.variable("a"), E.variable("n"))).create();
  assert(same(bad.validate({ core: true }), ["symbol 'a' needs a named reference object schema",
    "symbol 'b' needs a named reference object schema",
    "rule: 'pow' is not a core operation",
    "rule: argument 0: variable 'a' is not bound", // a symbol with a problem binds nothing
    "rule: argument 1: variable 'n' is not bound"]));
  assert(same(new P.OfPredicate.Builder().create().validate(), ["a predicate needs a rule"])); // a name and symbols are optional
  raises(Errors.ValueError, () => C.check([bad]), "predicate 'Bad': symbol 'a' needs a named reference object schema; predicate 'Bad'");
  raises(Errors.ValueError, () => C.check([IsAnAdult, IsAnAdult, (b: P.OfPredicate.Builder) => b.symbols({ x: Contact }).requires(true)]),
    "predicate 'IsAnAdult': defined twice; predicate None: a predicate of a set needs a name");
  raises(Errors.ValueError, () => C.check(new C.OfSet([the.data])), "a set's arguments are predicates");
  const predicates = C.check([IsAnAdult, (b: P.OfPredicate.Builder) => b.name("Named").symbols({ the: Contact }).requires(the.has("name"))]);
  assert(predicates instanceof C.OfSet && C.check(predicates) === predicates);
}

## CON-03 · A set orders predicates

In [ ]:
{
  const predicates = new C.OfSet.Builder().predicates(IsAnAdult).predicates(Numbered).create();
  assert(same(predicates.predicates, [IsAnAdult, Numbered])); // in order
  assert(new C.OfSet.Builder().create().predicates.length === 0 && predicates.schema_name() === "Patterns.Set");
  assert(predicates.owner() === null && predicates.identity() === predicates.identity()
    && predicates.identity() !== new C.OfSet.Builder().create().identity());
  const grown = new C.OfSet.Builder(predicates).predicates((b) => b.name("Grown").symbols({ the: Contact }).requires(true)).clone();
  assert(same(grown.predicates.map((p) => p.name), ["IsAnAdult", "Numbered", "Grown"]) && predicates.predicates.length === 2);
  assert(new C.OfSet.Builder(grown).predicates(IsAnAdult).update() === grown && grown.predicates.length === 4);
  raises(Errors.ValueError, () => new C.OfSet.Builder(predicates).create(),
    "create() is only valid without a source instance; use clone() or update()");
}

## CON-04 · Predicates are data: written by their symbols' schema names, read back through a store

In [ ]:
{
  const shared = the.age.ge(18n).data;
  const predicates = new C.OfSet.Builder().predicates(
    new P.OfPredicate.Builder().name("IsAnAdult").description("18 or older").symbols({ the: Contact }).requires(shared).create(),
    (b) => b.name("Owns").symbols({ c: Contact, p: Phone }).requires(E.operation("implies", shared, E.variable("p").has("number"))),
  ).create();
  const store = new C.OfStore(book()); // resolves the symbols' schemas in the address book
  const text = SchemaJSON.ToJSON(store).Reachable(C.OfSet.Schema, predicates);
  assert(text.includes('"symbols": [{"name": "c", "type": {"named": {"name": "Contact"}}}')); // by name: no store needed to write
  assert(text.split('"kind": "operation"').length - 1 === 4); // get, ge, implies and has: the shared rule's two are written once
  assert(SchemaJSON.ToJSON(C.Builders).Reachable(C.OfSet.Schema, predicates) === text); // any store writes them alike
  const copy = SchemaJSON.FromJSON(store).Reachable(C.OfSet.Schema, text) as P.OfSet;
  assert(JSON.stringify(copy.predicates.map((p) => [p.name, p.description])) === JSON.stringify([["IsAnAdult", "18 or older"], ["Owns", null]]));
  const owns = copy.predicates[1] as P.OfPredicate;
  assert(owns.symbols.get("c") === Contact && owns.symbols.get("p") === Phone); // the store's own
  assert(owns.rule.arguments[0] === copy.predicates[0].rule); // still shared
  assert(Terms.same(owns.rule, predicates.predicates[1].rule));
  assert(SV.Validate(store).Reachable(C.OfSet.Schema, predicates).length === 0); // valid data, by mbse-schemas' rules
  raises(Errors.LookupError, () => SchemaJSON.FromJSON(C.Builders).Reachable(C.OfSet.Schema, text),
    "no schema named 'Contact' in the module or the store"); // reading resolves names, so it needs the store
  const Inline = new S.OfObject.Builder().ref().properties((p) => p.name("n").of((t) => t.as_native(BigInt))).create();
  const loose = new P.OfPredicate.Builder().name("Loose").symbols({ x: Inline }).parameters((p) => p.name("k")).requires(true).create();
  const back = SchemaJSON.FromJSON(C.Builders).Reachable(P.OfPredicate.Schema,
    SchemaJSON.ToJSON(C.Builders).Reachable(P.OfPredicate.Schema, loose)) as P.OfPredicate;
  const x = back.symbols.get("x") as S.OfObject.Data;
  assert(x !== Inline && same(x.properties.keys(), ["n"])); // an unnamed schema, inline
  assert(equal([...back.parameters], [["k", null]])); // a parameter of any type
  const built = store.builder("Patterns.Predicate", IsAnAdult);
  built.property("name", (p: any) => p.value((a: any) => a.as_native((n: any) => n.set("Grown"))));
  assert(built.update() === IsAnAdult && IsAnAdult.name === "Grown" && IsAnAdult.symbols.get("the") === Contact);
  IsAnAdult.name = "IsAnAdult";
}

## CON-05 · Decoding refuses what a predicate or a set cannot hold

In [ ]:
{
  const store = new C.OfStore(book());
  const entry = (target: unknown, i: bigint) => (a: any) => a.add((e: any) => {
    if (target !== null) e.link("argument", (l: any) => l.set(target));
    e.property("index", (p: any) => p.value((v: any) => v.as_native((n: any) => n.set(i))));
  });
  raises(Errors.ValueError, () => store.builder("Patterns.Predicate").adjacency("arguments", entry(the.data, 1n)).create(),
    "a predicate's rule is argument 0, got index 1");
  raises(Errors.ValueError, () => store.builder("Patterns.Predicate").adjacency("arguments", entry(null, 0n)).create(),
    "link 'argument' is not set");
  const odd = store.builder("Patterns.Set").adjacency("arguments", entry(the.data, 0n)).create() as P.OfSet;
  assert(same(odd.predicates, [the.data]) && same(odd.check(), ["a set's arguments are predicates"])); // read, then refused
  const blank = store.builder("Patterns.Predicate").create() as P.OfPredicate; // what a snapshot leaves out is absent
  assert(blank.name === null && blank.description === null && blank.symbols.size === 0 && blank.parameters.size === 0 && blank.rule === null);
  assert(SchemaJSON.ToJSON(C.Builders)(P.OfPredicate.Schema, blank) === '{"root": "s0", "objects": {"s0": {"kind": "predicate"}}}');
}

## CON-06 · Another store holds predicates as proxies

In [ ]:
{
  const store = register(new Proxies.OfStore());
  for (const name of ["Patterns.Predicate", "Patterns.Set", "Patterns.OfApply", "Patterns.OfWeights", "Expressions.OfOperation"]) {
    assert(store.names().includes(name), name);
  }
  assert(register(store) === store); // what it holds already is skipped
  const loose = new P.OfPredicate.Builder().name("Any").symbols({ x: new S.OfObject.Builder().ref().create() }).requires(true).create();
  const proxy = SchemaJSON.FromJSON(store).Reachable(P.OfPredicate.Schema,
    SchemaJSON.ToJSON(C.Builders).Reachable(P.OfPredicate.Schema, loose)) as any;
  assert(proxy.schema_name() === "Patterns.Predicate" && Proxies.store_of(proxy) === store && proxy.name === "Any");
}

## CON-07 · A predicate is applied by reference, its arguments bound to its symbols and parameters

In [ ]:
{
  const HasName = new P.OfPredicate.Builder().name("HasName").symbols({ person: Contact }).parameters((p) => p.name("name"))
    .requires(E.variable("person").name.eq(E.variable("name"))).create();
  const Named = new P.OfPredicate.Builder().name("Named").symbols({ c: Contact });
  const Ann = Named.requires(HasName.call(Named.c, "ann")).create(); // the builder gives its symbols as variables
  const applied = Ann.rule as P.OfApply;
  assert(applied instanceof P.OfApply && applied.predicate === HasName); // the predicate itself
  assert(Terms.same(applied.arguments[0], E.variable("c").data) && applied.arguments[1].value === "ann");
  assert(Terms.same(HasName.call(E.variable("c"), "ann").arguments[1], applied.arguments[1])); // Python also calls it directly
  assert(Ann.validate({ core: true }).length === 0 && HasName.validate({ core: true }).length === 0);
  assert(same(P.DIALECT.validate(HasName.call(E.variable("c")), { bound: ["c"] }), ["'HasName' takes 2 arguments, got 1"]));
  assert(same(P.DIALECT.validate(new P.OfApply(the.data, []), { bound: ["the"] }), ["an application's predicate must be a predicate"]));
  assert(same(P.DIALECT.validate(new P.OfApply(null, [])), ["an apply needs a predicate"]));
  const store = book();
  const [ann, bob] = [store.Contact().name("ann").create(), store.Contact().name("bob").create()];
  const evaluate = new P.Evaluator(store);
  assert(P.holds(evaluate, Ann.rule, { c: ann }) === true && P.holds(evaluate, Ann.rule, { c: bob }) === false);
  assert(evaluate.run(HasName.call(E.variable("c"), E.variable("n")), { c: bob, n: "bob" }) === true);
  const shared = new C.OfSet.Builder().predicates(Ann,
    (b) => b.name("Bob").symbols({ c: Contact }).requires(HasName.call(E.variable("c"), "bob"))).create();
  const schemas = new C.OfStore(book());
  const text = SchemaJSON.ToJSON(schemas).Reachable(C.OfSet.Schema, shared);
  assert(text.split('"name": "HasName"').length - 1 === 1); // applied twice, written once
  const copy = SchemaJSON.FromJSON(schemas).Reachable(C.OfSet.Schema, text) as P.OfSet;
  assert(copy.predicates[0].rule.predicate === copy.predicates[1].rule.predicate
    && equal([...copy.predicates[0].rule.predicate.parameters], [["name", null]]));
  const cyclic = new P.OfPredicate.Builder().name("Loop").symbols({ c: Contact }).create();
  new P.OfPredicate.Builder(cyclic).requires(cyclic.call(E.variable("c"))).update();
  assert(same(P.DIALECT.validate(cyclic), ["rule: predicate: the expression contains a cycle"]));
}